# Example 11 - SA-CVA parity (FD vs CG/AAD) for EQ and COM

This notebook walks through two tiny SA-CVA portfolios (SP5 equity call, WTI quanto forward) to show how to align a finite-difference bump stack with the computation-graph AAD stack (with JIT and SIMD where available). Everything is self-contained under this folder.

- Goal: run both stacks on identical data and grids, then compare base CVA/EPE and sensitivities.
- Scope: EQ and COM spot/vol risk with IR, FX, and default curves for CPTY_A on an uncollateralised netting set.

## What lives here
- Master drivers: `Input/vre_sacva_cg_ad_eqcom.xml` (CG/AAD) and `Input/vre_sacva_fd_aad_eqcom.xml` (FD bump + AAD Greeks), both pointing to local `Input/sacva_*_eqcom/` trees.
- Portfolios: `portfolio_sacva_eqcom.xml` has one SP5 call and one WTI quanto forward, both on `CPTY_A`/`CPTY_A` netting.
- Pricing: `pricingengine_amccg.xml` carries the CG flags (domestic measure, midpoint, compact solve, lambda sweep list, FX-EQ drift gate); FD uses the classic engine set.
- Simulation/sensi: `simulation.xml`, `sensimarket.xml`, `xvasensiconfig.xml`, and `sensitivity.xml` are mirrored between stacks so grids/tenors match.
- Static data: `todaysmarket.xml`, `curveconfig.xml`, `market_20160205.txt`, `fixings_20160205.txt`, `netting.xml`, `counterparty.xml`, `collateralbalances.xml`.

## Performance and model levers (all config-driven)
- CG AAD: computation graph with pathwise adjoints; JIT and SIMD kick in automatically when available.
- Measure/scheme: domestic measure and midpoint evolution for EQ/COM/FX to stabilise quanto drift; compact solve vs lambda sweep for EQ loadings; optional FX-EQ drift gate for experiments.
- GPU: set `EXTERNAL_COMPUTE_DEVICE` to run on Metal/OpenCL/CUDA; scripts and utils handle the XML patching automatically.
- Parity hygiene: identical samples, time steps, tenors, and shift sizes across CG and FD to keep differences diagnostic.

## Config wiring at a glance
```
vre_sacva_*_eqcom.xml (CG or FD)
  |- pricingengine*.xml   -> CG flags (domestic measure, midpoint, compact/lambda, drift gate)
  |- todaysmarket.xml     -> market_20160205.txt, fixings_20160205.txt, dividends
  |- curveconfig.xml      -> discount/forecast/default/equity/commodity curve IDs
  |- simulation.xml       -> samples, time steps per year, valuation grid
  |- sensimarket.xml      -> simulated IR 1Y/5Y, credit 6M/1Y/5Y, FX/EQ/COM vols
  |- xvasensiconfig.xml   -> shift sizes/tenors for AAD grid (credit 6M/1Y/5Y, IR 1Y/5Y)
  |- sensitivity.xml      -> FD bump sizes/tenors (mirrors xvasensiconfig)
  |- portfolio/netting/counterparty/collateral
```

## Worked mapping: equity and commodity chains (alignment matters)

A tidy wiring avoids silent skews. Keep market/fixing files identical, mirror grids, and point both stacks at the same IDs before you compare CVA or deltas.

**Alignment checklist**
- Same `market_20160205.txt` + `fixings_20160205.txt` for CG and FD
- Matching curve/vol grids: IR 1Y/5Y, credit 6M/1Y/5Y, EQ/COM vol expiries
- Pricing flags aligned: domestic measure, midpoint, compact/lambda list, drift gate

**Equity SP5 call (USD pay, EUR base)**

[todaysmarket]  SP5 -> Equity/USD/SP5, vol -> EquityVolatility/USD/SP5, FX -> FX/EUR/USD

[curveconfig]   EQ-SP5 discounted on USD-FED; EUR reporting via EURUSD

[sensimarket]   dividends 6M/1Y/2Y; EQ vols 6M/5Y/10Y; FX vols 1Y/5Y

[sensi/xvasensi] EQ spot 1% rel; EQ vol 1% rel; dividend 1e-6 abs; IR 1Y/5Y; credit 6M/1Y/5Y

[pricingengine] domestic measure + midpoint; compact=true (else lambda sweep list); drift gate=false


XML anchors:
```xml
<EquityCurves id="default">
  <EquityCurve name="SP5">Equity/USD/SP5</EquityCurve>
</EquityCurves>
<EquityVolatilities id="default">
  <EquityVolatility name="SP5">EquityVolatility/USD/SP5</EquityVolatility>
</EquityVolatilities>
<EquityCurve id="EQ-SP5">
  <DiscountCurve>USD-FED</DiscountCurve>
  <ForwardQuote>Equity/USD/SP5</ForwardQuote>
</EquityCurve>
<EquityVolatility>
  <Name>SP5</Name>
  <Expiries>6M,5Y,10Y</Expiries>
</EquityVolatility>
```

**Commodity WTI quanto forward (USD pay, EUR base)**

[todaysmarket]  COMDTY_WTI_USD -> Commodity/USD/WTI_USD; vol -> CommodityVolatility/USD/WTI_USD_VOLS; FX EURUSD

[curveconfig]   WTI curve pillars ~1Y/5Y/10Y; USD discount; EUR reporting via FX

[sensimarket]   COM curve + vol at 1Y/5Y/10Y (moneyness 0,1); FX vols 1Y/5Y; credit 6M/1Y/5Y

[sensi/xvasensi] COM spot/vol 1% rel; FX spot/vol 1% rel; IR/credit 1 bp abs

[pricingengine] domestic measure + midpoint + compact solve (lambda sweep list available)

XML anchors:
```xml
<CommodityCurves id="default">
  <CommodityCurve name="COMDTY_WTI_USD">Commodity/USD/WTI_USD</CommodityCurve>
</CommodityCurves>
<CommodityVolatilities id="default">
  <CommodityVolatility name="COMDTY_WTI_USD">CommodityVolatility/USD/WTI_USD_VOLS</CommodityVolatility>
</CommodityVolatilities>
<Commodities>
  <Simulate>true</Simulate>
  <Names><Name>COMDTY_WTI_USD</Name></Names>
  <Tenors>1Y,5Y,10Y</Tenors>
</Commodities>
<CommodityVolatilities>
  <Names>
    <Name id="COMDTY_WTI_USD">
      <Expiries>1Y,5Y,10Y</Expiries>
      <Moneyness>0.0,1.0</Moneyness>
    </Name>
  </Names>
</CommodityVolatilities>
```

## Helper paths

In [1]:
from pathlib import Path
from utils import describe_master_portfolio

try:
    EXAMPLE_DIR = Path(__file__).resolve().parent  # when run via python
except NameError:
    EXAMPLE_DIR = Path.cwd()  # when run in notebook
CG_MASTER = EXAMPLE_DIR / "Input" / "vre_sacva_cg_ad_eqcom.xml"
FD_MASTER = EXAMPLE_DIR / "Input" / "vre_sacva_fd_aad_eqcom.xml"
CG_INPUT = EXAMPLE_DIR / "Input" / "sacva_cg_eqcom"
FD_INPUT = EXAMPLE_DIR / "Input" / "sacva_fd_eqcom"
CG_OUTPUT = EXAMPLE_DIR / "Output" / "sacva_cg_aad_eqcom"
FD_OUTPUT = EXAMPLE_DIR / "Output" / "sacva_fd_aad_eqcom"

print("CG master:", CG_MASTER)
print("FD master:", FD_MASTER)
print("CG input:", CG_INPUT)
print("FD input:", FD_INPUT)
print("CG output:", CG_OUTPUT)
print("FD output:", FD_OUTPUT)

CG master: /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Input/vre_sacva_cg_ad_eqcom.xml
FD master: /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Input/vre_sacva_fd_aad_eqcom.xml
CG input: /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Input/sacva_cg_eqcom
FD input: /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Input/sacva_fd_eqcom
CG output: /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Output/sacva_cg_aad_eqcom
FD output: /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Output/sacva_fd_aad_eqcom


## Portfolio snapshot

In [2]:
describe_master_portfolio(CG_MASTER)
describe_master_portfolio(FD_MASTER)

Portfolio file: /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Input/sacva_cg_eqcom/portfolio_sacva_eqcom.xml
Trades: 2 | By type:  ScriptedTrade=2
Currencies: 
Netting sets: CPTY_A
CSA definition file: /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Input/sacva_cg_eqcom/netting.xml
Collateral balances file: /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Input/sacva_cg_eqcom/collateralbalances.xml
Portfolio file: /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Input/sacva_fd_eqcom/portfolio_sacva_eqcom.xml
Trades: 2 | By type:  ScriptedTrade=2
Currencies: 
Netting sets: CPTY_A


## How the SA-CVA CG pipeline flows

```
[1] Config load
    master XML -> pricingengine_amccg.xml -> todaysmarket.xml -> curveconfig.xml
    simulation.xml + sensimarket.xml + xvasensiconfig.xml + sensitivity.xml
      |
[2] Market build
    quotes/fixings -> Market + ScenarioGenerator (IR/FX/EQ/COM/default grids)
      |
[3] Factor mapping
    maps trade risk keys to simulated factors (spot/vol/curve IDs, tenors, buckets)
      |
[4] CG build
    ScriptedTrade graph -> GaussianCamCG (measure/midpoint/compact/lambda)
      |
[5] Forward pass
    MC paths -> NPVs/exposure per scenario/date; CAM writes exposure cube
      |
[6] Reverse pass (AAD)
    adjoints on the same paths -> pathwise Greeks -> aggregated sensi cube
      |
[7] SA-CVA aggregation
    cva_sensitivities.csv + sacva_sensitivities.csv (credit/IR/EQ/COM/FX)
    sacva.csv (capital), xva_exposure.csv (EPE/ENE), cg_trace*.csv (debug)
```

**SaaS / downstream**
```
Outputs -> upload
  sacva.csv, sacva_sensitivities.csv, cva_sensitivities.csv, xva_exposure.csv
    |
  BigQuery landing tables
    |
  Post-processing: API surfaces, raw SQL, AI Q&A, trend/risk alerts
```

Alignment tip: every arrow depends on consistent IDs/tenors; if grids drift, the CG sensi cube and SA-CVA aggregation diverge.

## Run both examples from here

Use the cells below to execute the CG/AAD and FD runs directly from the notebook (uses the current Python kernel and venv). Set `USE_GPU = True` to patch the XMLs for your GPU via `EXTERNAL_COMPUTE_DEVICE` if desired.

In [3]:
import os, subprocess, sys
from pathlib import Path

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()
USE_GPU = False  # set True and optionally export EXTERNAL_COMPUTE_DEVICE


def run_case(script: str, extra_env=None):
    env = os.environ.copy()
    if extra_env:
        env.update({k: v for k, v in extra_env.items() if v is not None})
    cmd = [sys.executable, str(NOTEBOOK_DIR / script)]
    print(">>>", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=NOTEBOOK_DIR, env=env, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode:
        raise RuntimeError(f"{script} failed with code {proc.returncode}")

extra = {}
if USE_GPU:
    # EXTERNAL_COMPUTE_DEVICE can be pre-set; leave blank to auto-detect in scripts/utils
    extra["EXTERNAL_COMPUTE_DEVICE"] = os.environ.get("EXTERNAL_COMPUTE_DEVICE", "")
    extra["USE_EXTERNAL_COMPUTE_DEVICE"] = "true"

run_case("run_sacva_eqcom.py", extra_env=extra)
run_case("run_sacva_eqcom_fd_aad.py", extra_env=extra)

>>> /Volumes/Fast4/Vannarho/vre/venv/bin/python3.14 /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/run_sacva_eqcom.py


VRE executable not found.

>>> /Volumes/Fast4/Vannarho/vre/venv/bin/python3.14 /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/run_sacva_eqcom_fd_aad.py


VRE executable not found.



## Inspect CG outputs

List the key files produced by the CG run and preview the first rows inline.

In [4]:
from pathlib import Path
import pandas as pd

cg_dir = CG_OUTPUT  # set earlier
print("CG output dir:", cg_dir)
files = [
    ("sacva.csv", "SA-CVA capital summary"),
    ("sacva_sensitivities.csv", "SA-CVA sensitivities"),
    ("cva_sensitivities.csv", "CAM CVA sensitivities"),
    ("xva_exposure.csv", "Exposure profile (EPE/ENE)")
]
available = []
for name, desc in files:
    p = Path(cg_dir) / name
    status = "FOUND" if p.exists() else "missing"
    print(f"- {name:25s} {status:7s} : {desc}")
    if p.exists():
        available.append(p)
for extra in sorted(Path(cg_dir).glob("cg_trace*.csv")):
    print(f"- {extra.name:25s} FOUND    : Sensi trace (debug)")
    available.append(extra)

TARGET = available[0] if available else None  # pick first available; change manually if desired
if TARGET:
    print("\nPreviewing:", TARGET.name)
    if TARGET.suffix.lower() == ".csv":
        display(pd.read_csv(TARGET).head(20))
    else:
        print(TARGET.read_text()[:2000])
else:
    print("\nRun the CG job to populate outputs.")

CG output dir: /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Output/sacva_cg_aad_eqcom
- sacva.csv                 missing : SA-CVA capital summary
- sacva_sensitivities.csv   missing : SA-CVA sensitivities
- cva_sensitivities.csv     missing : CAM CVA sensitivities
- xva_exposure.csv          missing : Exposure profile (EPE/ENE)

Run the CG job to populate outputs.


## Quick diff on outputs (run after the jobs finish)

In [5]:
import pandas as pd
from pathlib import Path

names = ["sacva.csv", "sacva_sensitivities.csv", "xva_exposure.csv", "cva_sensitivities.csv"]

for name in names:
    cg_path = CG_OUTPUT / name
    fd_path = FD_OUTPUT / name
    print(f"\n{name}")
    if not cg_path.exists() or not fd_path.exists():
        print("  run the jobs to create both outputs (", cg_path, ", ", fd_path, ")")
        continue
    try:
        cg = pd.read_csv(cg_path)
        fd = pd.read_csv(fd_path)
    except Exception as e:
        print("  could not read CSV:", e)
        continue
    if set(cg.columns) != set(fd.columns):
        print("  column mismatch; showing CG columns only")
        print(sorted(cg.columns))
        continue
    # align rows by common keys if present
    keys = [k for k in ["TradeId", "Type", "Bucket", "Factor"] if k in cg.columns]
    if keys:
        merged = cg.merge(fd, on=keys, suffixes=("_cg", "_fd"))
        num_cols = [c for c in merged.columns if c.endswith("_cg")]
        if not num_cols:
            print("  no numeric columns to compare")
            continue
        diffs = []
        for col in num_cols:
            peer = col[:-3] + "_fd"
            if peer in merged:
                delta = (merged[col] - merged[peer]).abs()
                diffs.append((col[:-3], float(delta.max()), float(delta.mean())))
        if diffs:
            diffs.sort(key=lambda x: x[1], reverse=True)
            print("  max/mean abs diff per column:")
            for name_, mx, mean in diffs:
                print(f"    {name_:20s} max={mx:.4g} mean={mean:.4g}")
        else:
            print("  nothing to compare")
    else:
        delta = (cg.select_dtypes(include=float) - fd.select_dtypes(include=float)).abs()
        print(delta.describe())


sacva.csv
  run the jobs to create both outputs ( /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Output/sacva_cg_aad_eqcom/sacva.csv ,  /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Output/sacva_fd_aad_eqcom/sacva.csv )

sacva_sensitivities.csv
  run the jobs to create both outputs ( /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Output/sacva_cg_aad_eqcom/sacva_sensitivities.csv ,  /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Output/sacva_fd_aad_eqcom/sacva_sensitivities.csv )

xva_exposure.csv
  run the jobs to create both outputs ( /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Output/sacva_cg_aad_eqcom/xva_exposure.csv ,  /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Output/sacva_fd_aad_eqcom/xva_exposure.csv )

cva_sensitivities.csv
  run the jobs to create both outputs ( /Volumes/Fast4/Vannarho/vre/Examples/VREPython/Notebooks/Example_11/Output/sacva_c

## Reading residual differences
- Credit and IR: both grids use credit tenors 6M/1Y/5Y and IR tenors 1Y/5Y; small gaps usually come from CAM recalibration vs FD bumping. Increase samples/time steps if variance dominates.
- EQ/COM vegas: CG relies on vol surfaces in `sensimarket.xml`; if you toggle sim vols off, vegas will drop to zero. Keep EQ/COM vols on for parity runs.
- FX orientation and CSA: this example is uncollateralised with EUR base currency and EURUSD orientation; changing CSA or base currency will move CVA levels.
- Measure/scheme: domestic + midpoint + compact solve are the default parity set; try the lambda sweep or drift gate only when debugging model differences.